In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("ps06.ipynb")

# PS6 — Noise, Gates and Logic in Time
### BioE 147/247 · Fall 2026

**Out:** Tuesday, October 20 · **Due:** Wednesday, October 29, 11:59 pm
**Covers:** Sessions 12, 13 and 16 · **48 points** · **Question 8** is required for **BioE 247** (56 total) and is extra credit for **BioE 147** (up to +8)

---

Three sessions and one thread: a circuit is not its steady state.

Session 12 said a mean is not a measurement — two cells with the same average
behave differently, and you need a distribution. Session 13 said a gate is not
a curve — it is four numbers, and whether two gates connect is arithmetic you
can do before you build anything. Session 16 said a truth table is not a
circuit — it tells you where the output ends up and nothing about what it does
on the way.

Questions 1–4 are the first, 5 the second, 6–7 the third.

Two of these are items your handouts said would be here. They are the same
items: the RBS window from session 13, and the fail-safe circuit that still
glitches from session 16.

**The tick convention for Questions 6 and 7**, which is printed on the session
16 handout:

> Time runs in ticks. At each tick every gene reads its inputs **as they were
> at the previous tick** and updates; all genes update together. At $t = 0$
> the cell has just been made, so **every gene is at 0**.

**Collaboration is encouraged.** Discuss, argue, work at a whiteboard
together, then write your own solution and your own code. Record who you
worked with below.

**If you used an LLM**, say so briefly and say what for. The conditions are
that you can explain anything you submit and that the code you submit runs.

**A note on the visible tests.** They check *properties* — symmetries, counts,
what happens at a limit. A green visible check means "not obviously broken",
not "right".

**A note on the stochastic questions.** Your numbers will not match these
exactly, and they are not supposed to. The tests allow the spread that a
finite run actually has. If a test fails by a hair, run longer before you
change your code.

In [ ]:
COLLABORATORS = ""   # e.g. "worked with J. Chen on Q6"
AI_USE = ""          # e.g. "used an LLM to debug the SSA loop"

## Setup

In [ ]:
# ---------------------------------------------------------------------------
# SETUP — run this cell first, every time.
#
# DataHub / local : finds the repository root and puts it on the import path.
# Google Colab    : clones the repository first, because Colab opens this
#                   notebook on its own, without the posb package beside it.
# ---------------------------------------------------------------------------
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.exists("posb2026"):
        !git clone -q https://github.com/ArkinLaboratory/posb2026.git
    sys.path.insert(0, os.path.abspath("posb2026"))
else:
    _d = os.getcwd()
    while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "posb")):
        _d = os.path.dirname(_d)
    sys.path.insert(0, _d)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import posb
from posb import Reaction, Model

posb.check_environment()


In [ ]:
from posb.stochastic import (time_average, fano, cv,
                             birth_death, negative_autoregulation,
                             nar_fano_lna)
from posb.digital import Repressor
from posb.logic import (GateNetwork, NOT, BUF, NAND, NOR, HYBRID,
                        hazards)
from scipy.optimize import brentq

---
## Question 1 — The master equation, by hand  *(4 points)*

The simplest unit in the course: one species $X$, made at a constant rate $k$
and degraded at rate $\gamma$ per molecule.

$$\varnothing \xrightarrow{\;k\;} X, \qquad X \xrightarrow{\;\gamma x\;}
\varnothing$$

<!-- BEGIN QUESTION -->

**Q1.** In the cell below, in **writing** (no code):

**(a)** Write the chemical master equation for $P(x, t)$, the probability of
having exactly $x$ molecules at time $t$. Four terms, and say in one phrase
what each one is.

**(b)** Set $dP/dt = 0$ and show that the stationary distribution is Poisson
with mean $k/\gamma$. You may do this by verifying that the Poisson satisfies
detailed balance between $x$ and $x+1$ — which is the one-line route, and
saying *why* that route is available here is part of the answer.

**(c)** State the Fano factor and the CV of that distribution, in terms of
$k$ and $\gamma$. One sentence on why the CV falls as the mean rises, and what
that implies for a cell trying to make a decision on a protein present at ten
copies.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 2 — A Gillespie SSA, from scratch  *(8 points)*

`posb.stochastic.gillespie` exists. **Do not use it in Q2a.** Write the loop
yourself; the point of this question is that you have written one once.

The algorithm, as given in class:

1. From the current state, compute every propensity $a_j$ and their sum $a_0$.
2. Draw the waiting time $\tau \sim \text{Exponential}(a_0)$, i.e.
   $\tau = -\ln(u_1)/a_0$ with $u_1$ uniform on $(0,1]$.
3. Choose reaction $j$ with probability $a_j/a_0$.
4. Apply its stoichiometry, advance $t$ by $\tau$, record, repeat.

**Q2a.** Write `ssa(stoich, propensity, x0, t_max, rng)` returning
`(t, X)` — a 1-D array of event times starting at 0.0, and a 2-D array of
states with one **row per recorded time**. Stop when $t$ would exceed
`t_max`, and stop if $a_0 = 0$.

Signature and semantics match `posb.stochastic.gillespie`, so you can check
yourself against it in Q2b.

In [ ]:
def ssa(stoich, propensity, x0, t_max, rng):
    """A direct-method Gillespie SSA. Returns (t, X)."""
    ...


stoich, prop = birth_death(20.0, 0.2)
t, X = ssa(stoich, prop, [0], 200.0, np.random.default_rng(0))
print(f"{len(t)} events, final state {X[-1, 0]}")

In [ ]:
grader.check("q2a")

**Q2b.** Run your SSA on the constitutive unit with $k = 20$,
$\gamma = 0.2$ (so the mean should be 100), from $x_0 = 100$, to
$t_{\max} = 2000$, discarding the first 200 time units as burn-in. Use
`time_average`, which weights by the time spent in each state — **not**
`X.mean()`, which weights by events and is biased.

Set `MEAN`, `VAR`, `FANO`, `CV_X` to your four numbers.

Then say, in the markdown cell that follows, what the Fano factor tells you
and how close to 1 you would need it to be before you believed the unit was
Poisson.

In [ ]:
t, X = ssa(*birth_death(20.0, 0.2), [100], 2000.0,
           np.random.default_rng(0))
...
print(f"mean {MEAN:.2f}   var {VAR:.2f}   Fano {FANO:.3f}   CV {CV_X:.4f}")

In [ ]:
grader.check("q2b")

---
## Question 3 — Does negative autoregulation actually reduce the noise?  *(7 points)*

Session 7 asserted that a gene repressing its own promoter is quieter than a
constitutive one. The ODE you were integrating at the time **cannot produce a
variance at all**, so that was a claim, not a result. You can now check it.

The comparison only means something at **matched mean**. A circuit that is
quieter because it makes fewer molecules has told you nothing.

**Q3a.** With $\gamma = 0.2$, $K = 100$ and $n = 4$, find the
$\beta$ that puts the deterministic steady state at exactly 100 molecules, and
set `BETA` to it. Then run both units from $x_0 = 100$ to $t_{\max} = 4000$
with burn-in 400, and set

`FANO_CONST`, `CV_CONST`, `FANO_NAR`, `CV_NAR`.

Use `posb.stochastic.gillespie` here — you wrote your own in Q2 and that debt
is paid.

In [ ]:
from posb.stochastic import gillespie

GAMMA, K_NAR, N_NAR = 0.2, 100.0, 4.0

...

print(f"beta = {BETA:.3f}")
print(f"constitutive  Fano {FANO_CONST:.3f}   CV {CV_CONST:.4f}")
print(f"NAR, n = 4    Fano {FANO_NAR:.3f}   CV {CV_NAR:.4f}")

In [ ]:
grader.check("q3a")

<!-- BEGIN QUESTION -->

**Q3b.** Two short paragraphs.

**(a)** `posb.stochastic.nar_fano_lna(n, x/K)` returns the linear-noise
prediction $1/(1+g)$, where $g = n u^n/(1+u^n)$ is the logarithmic sensitivity
of the production rate at the operating point — the **same loop gain** that
set the repressilator's boundary in session 11. Evaluate it for your case and
compare with your measured Fano. Why is $g$ the quantity that matters, rather
than $n$ or $K$ separately?

**(b)** Becskei & Serrano (*Nature* 2000, Fig. 3a) measured this. Your
simulation has one source of noise; a cell has more. Name one noise source
present in their experiment and absent from your model, and say whether it
would make the measured improvement larger or smaller than yours.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 4 — Bursting  *(5 points)*

Proteins are not made one at a time. A transcript appears, is translated
$b$ times on average, and is degraded; the protein arrives in **bursts**.

$$\varnothing \xrightarrow{k_m} m, \qquad
m \xrightarrow{\gamma_m m} \varnothing, \qquad
m \xrightarrow{k_p m} m + p, \qquad
p \xrightarrow{\gamma_p p} \varnothing$$

**Q4.** Write two functions of the rate constants alone — no
simulation:

- `burst_size(k_p, gamma_m)` — the mean number of proteins made per transcript.
- `burst_frequency(k_m, gamma_p)` — the number of transcripts per protein
  lifetime.

Then write `protein_fano(k_p, gamma_m, gamma_p)` returning the stationary Fano
factor of the protein. Derive it rather than importing it; you may check
against `two_stage_fano` afterwards.

Finally, set `MEAN_P` and `FANO_P` for $k_m = 0.5$, $\gamma_m = 1$,
$k_p = 10$, $\gamma_p = 0.1$, **from your formulas**, and confirm against one
simulation.

In [ ]:
K_M, GAMMA_M, K_P, GAMMA_P = 0.5, 1.0, 10.0, 0.1


def burst_size(k_p, gamma_m):
    """The mean number of proteins made from one transcript."""
    ...


def burst_frequency(k_m, gamma_p):
    """Transcripts per protein lifetime."""
    ...


def protein_fano(k_p, gamma_m, gamma_p):
    """The stationary Fano factor of the protein, from the rates alone."""
    ...


...

print(f"burst size {burst_size(K_P, GAMMA_M):.1f} proteins per transcript")
print(f"burst frequency {burst_frequency(K_M, GAMMA_P):.1f} per protein lifetime")
print(f"mean {MEAN_P:.1f}   Fano {FANO_P:.3f}")

In [ ]:
grader.check("q4")

---
## Question 5 — The gate datasheet, and whether two gates connect  *(9 points)*

A repressor gate, input $x$ and output $y$, both as concentrations:

$$y(x) = y_{\min} + \frac{y_{\max}-y_{\min}}{1+(x/K)^{n}}$$

Its gain is the **log slope**, $G = d\ln y / d\ln x$. The two solutions of
$|G| = 1$ are the input thresholds $x_{IL} < x_{IH}$, and they name the
worst-case output levels $y_{OH} = y(x_{IL})$ and $y_{OL} = y(x_{IH})$. Those
four numbers are the datasheet.

Two gates, both with $y_{\min} = 0.1$, $y_{\max} = 10$, $K = 1$:
**gate A** has $n = 2$, **gate B** has $n = 3$.

**Q5a.** Write `datasheet(gate)` returning the four numbers as a
dict with keys `"x_IL"`, `"x_IH"`, `"y_OH"`, `"y_OL"`. Use
`Repressor.thresholds()` for the roots, but compute the two output levels
yourself from the transfer curve, so that you know which is which.

Set `SHEET_A` and `SHEET_B`.

**Watch the ordering.** `Repressor.output_levels()` returns $(y_{OL},
y_{OH})$, low first. Getting this backwards is the single most common error in
this question and it makes Q5b come out with the wrong sign.

In [ ]:
A = Repressor(0.1, 10, 1.0, 2, name="A")
B = Repressor(0.1, 10, 1.0, 3, name="B")


def datasheet(gate):
    """The four numbers that decide whether this gate connects to another."""
    ...


SHEET_A, SHEET_B = datasheet(A), datasheet(B)
for name, sh in (("A", SHEET_A), ("B", SHEET_B)):
    print(name, {k: round(v, 4) for k, v in sh.items()})

In [ ]:
grader.check("q5a")

**Q5b.** Gate A drives gate B. Write `margins(sender, receiver)`
returning $(\mathrm{NM_L}, \mathrm{NM_H})$ in **decades**:

$$\mathrm{NM_L} = \log_{10}\frac{x_{IL}(B)}{y_{OL}(A)}, \qquad
\mathrm{NM_H} = \log_{10}\frac{y_{OH}(A)}{x_{IH}(B)}$$

Both must be positive for the pair to connect. Set `NM_AB` to the pair, and
`CONNECTS` to `True` or `False`.

In [ ]:
def margins(sender, receiver):
    """Low and high noise margins, in decades."""
    ...


NM_AB = margins(A, B)
CONNECTS = bool(NM_AB[0] > 0 and NM_AB[1] > 0)
print(f"NM_L = {NM_AB[0]:+.4f}   NM_H = {NM_AB[1]:+.4f}   connects: {CONNECTS}")

In [ ]:
grader.check("q5b")

**Q5c.** The pair fails on the high side by a hair. Scaling gate A's
output by a factor $f$ — a stronger or weaker RBS — multiplies **both** of its
output levels by $f$ and moves **neither** gate's thresholds.

Find the range of $f$ for which both margins are positive. Set `F_LO` and
`F_HI` to the endpoints, to three decimals.

`Repressor.scaled(f)` gives you the scaled gate. Solve for the endpoints
rather than scanning: each margin is monotone in $f$, so each endpoint is one
root.

In [ ]:
...

print(f"the RBS window is f in [{F_LO:.3f}, {F_HI:.3f}]")
print("  check:", [round(x, 4) for x in margins(A.scaled((F_LO + F_HI) / 2), B)])

In [ ]:
grader.check("q5c")

---
## Question 6 — A truth table is not a circuit  *(11 points)*

A two-input classifier for a cell-killing actuator. The specification:

> fire **only** when surface marker $A$ is present **and** marker $B$ is
> absent.

Everything below uses the tick convention printed at the top of this set, and
`posb.logic` for the simulation.

<!-- BEGIN QUESTION -->

**Q6a.** In writing:

**(a)** The truth table for the specification. Four rows.

**(b)** A circuit that realises it, using only NOT gates and gates built on a
single promoter with two operators. Write it as a list of genes, each line
saying what represses what, and say which gene is the output.

**(c)** One sentence: a classmate hands in a different list that computes the
same table. What would make one of the two preferable, and what would make the
difference irrelevant?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Q6b.** Here is the circuit that was actually built, as a
`GateNetwork`. $R_2$ is one gene behind two independently repressed
promoters, so it is a NAND.

Write `trace(net, a, b, ticks=5)` returning the output's value at each tick as
a **list of ints**, and `glitch_report(net)` returning a dict mapping each
`(a, b)` pair to the hazard kind as a string, or `"clean"`.

Set `REPORT` to `glitch_report(built)`.

In [ ]:
built = GateNetwork(
    [NOT("R1", "B"), NAND("R2", "A", "R1"), NOT("R3", "R2")],
    inputs=("A", "B"), output="R3")


def trace(net, a, b, ticks=5):
    """The output's value at each tick, from an empty cell."""
    ...


def glitch_report(net, ticks=5):
    """{(a, b): hazard kind or "clean"} over all four input cases."""
    ...


REPORT = glitch_report(built)
for k, v in sorted(REPORT.items()):
    print(f"A={k[0]} B={k[1]}: {trace(built, *k)}  -> {v}")

In [ ]:
grader.check("q6b")

**Q6c.** Two circuits, both realising the same truth table as
`built`. Three properties, all in `posb.logic`:

- `net.is_fail_safe()` — the output gene is off one tick after the cell is
  made, i.e. its promoter reads 0 while every protein is still absent.
- `net.is_balanced()` — every route from any input to the output has the same
  number of genes.
- `net.cold_start_safe()` — a *sufficient* condition for never showing a wrong
  value from a cold start. Read its docstring **after** you have done (a).

A `HYBRID` gate takes `activators` and `repressors` and is ON only when every
activator is present and no repressor is; `BUF` is a one-input hybrid with a
single activator.

**(a)** Set `STILL_BAD` to a network that **is fail-safe and still glitches**.
Fail-safety alone does not save you, and one such circuit was on the slide.

**(b)** Set `CLEAN` to a network that glitches in **no** input case. It does
not have to be balanced, and the smallest answer is two genes.

In class we also saw a circuit that is fail-safe *and* balanced and glitches
anyway, so neither property is what you are aiming at. Part (b) is asking you
to find the property that is.

In [ ]:
...

for name, net in [("STILL_BAD", STILL_BAD), ("CLEAN", CLEAN)]:
    print(f"{name}:  fail-safe {net.is_fail_safe()}   "
          f"balanced {net.is_balanced()}   "
          f"cold_start_safe {net.cold_start_safe()}")
    print(f"   paths  A: {net.path_lengths('A')}   B: {net.path_lengths('B')}")
    for k, v in sorted(glitch_report(net).items()):
        print(f"     A={k[0]} B={k[1]}: {trace(net, *k)}  -> {v}")

In [ ]:
grader.check("q6c")

<!-- BEGIN QUESTION -->

**Q6d.** Three short paragraphs, using **your own circuits and
your own numbers from Q6b and Q6c**.

**(a)** The first answer everyone gives for Q6b is "the two paths are
different lengths". Say why that diagnosis is wrong, citing the balanced
circuit from class — and note that your own `CLEAN` is unbalanced.

**(b)** Say why "the output is OFF in an empty cell" is not enough either,
citing `STILL_BAD` and the cases it glitches in.

**(c)** Your `CLEAN` has a property neither of the others has. Say what it
is, in terms of **which site a late signal arrives at**. Then say what your
four circuits do **not** establish.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 7 — How late is a deep circuit?  *(4 points)*

A chain of $n$ genes, each driving the next, every protein sharing a lifetime
$\tau$, with a step at the input at $t = 0$. The output of stage $n$ is

$$y_n(t) = 1 - e^{-t/\tau} \sum_{k=0}^{n-1} \frac{(t/\tau)^k}{k!}$$

and the half-time $t_{1/2}(n)$ is where $y_n = 1/2$.

**Q7.** Write `half_time(n)` returning $t_{1/2}(n)$ in lifetimes, by
root-finding on the expression above. Do not call `cascade_half_time`; you may
check against it.

Then set `RULE_ERR` to a list of $|t_{1/2}(n) - (n - 1/3)|$ for
$n = 1 \ldots 5$, and `PER_STAGE` to the list of increments
$t_{1/2}(n) - t_{1/2}(n-1)$ for $n = 2 \ldots 5$.

In [ ]:
from math import factorial, exp


def half_time(n, tau=1.0):
    """Where the n-stage step response crosses half of its final value."""
    ...


...

for n in range(1, 6):
    print(f"n={n}  t_half = {half_time(n):.4f}   rule = {n - 1/3:.4f}")
print("per stage:", [round(v, 4) for v in PER_STAGE])

In [ ]:
grader.check("q7")

---
## Question 8 — required for BioE 247, extra credit for BioE 147

**BioE 247** — part of the assignment, worth 8 of your 56
points.

**BioE 147** — optional, worth up to 8 points of extra credit on top of
48. You do not need it for full marks. It is not a harder version of the
same thing; it removes an assumption the earlier questions made, which is
where most of the interest is.

Every variance in this set has been **intrinsic**: it came from the
randomness of the reactions themselves. A real cell also varies in things that
affect *every* gene at once — ribosome and polymerase abundance, plasmid copy
number, volume, where it is in its cycle. Those are **extrinsic**, and a
single-colour experiment cannot tell them apart from intrinsic noise, because
both just make the number different from cell to cell.

Elowitz, Levine, Siggia & Swain (*Science* 2002) separated them with two
copies of the same promoter driving two different fluorescent proteins in the
same cell. With $c$ and $y$ the two readouts, normalised to mean 1:

$$\eta^2_{\text{int}} = \frac{\langle (c-y)^2 \rangle}{2}, \qquad
\eta^2_{\text{ext}} = \langle c y \rangle - \langle c \rangle \langle y \rangle,
\qquad \eta^2_{\text{tot}} = \eta^2_{\text{int}} + \eta^2_{\text{ext}}.$$

<!-- BEGIN QUESTION -->

**Q8 (a).** Write `decompose(c, y)` taking two arrays of per-cell
readouts and returning `(eta2_int, eta2_ext, eta2_tot)`, normalising each
channel to its own mean first.

Then generate a synthetic two-colour population and check it recovers what you
put in: `simulate(n_cells, eta_int, eta_ext, rng)` should produce two channels
sharing one extrinsic factor and carrying independent intrinsic noise. Set
`REC_INT` and `REC_EXT` to the recovered values for
`eta_int = 0.25, eta_ext = 0.15, n_cells = 200000`.

In [ ]:
def decompose(c, y):
    """Intrinsic, extrinsic and total noise, Elowitz 2002 eq. 1-3."""
    ...


def simulate(n_cells, eta_int, eta_ext, rng):
    """Two channels, one shared extrinsic factor, independent intrinsic noise."""
    ...


c, y = simulate(200_000, 0.25, 0.15, np.random.default_rng(0))
...
print(f"asked for  int 0.0625   ext 0.0225")
print(f"recovered  int {REC_INT:.4f}   ext {REC_EXT:.4f}")
print("The intrinsic term need not come back as exactly 0.0625: a")
print("multiplicative model returns eta_int^2 (1 + eta_ext^2) = 0.0639,")
print("an additive one 0.0625. Both are accepted -- the question does")
print("not fix the noise model.")

**Q8 (b)-(c).** Two paragraphs, and they are the point of the question.

**(b)** Go back to Q3. You measured a CV for the constitutive unit and a CV
for the autoregulated one at matched mean, and concluded that autoregulation
reduces noise. **Which of the two noise components did that experiment
measure, and which could it not see?** Say what would have to be added to your
simulation for the other one to appear.

**(c)** Suppose a two-colour experiment on a real autoregulated promoter
returned $\eta^2_{\text{int}} = 0.004$ and $\eta^2_{\text{ext}} = 0.021$.
A colleague proposes to reduce the cell-to-cell variability by raising the
Hill coefficient of the autoregulation. Say whether that will work and why,
in terms of the two numbers.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

